In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

# ============================================================
# ROADFLOOD-VLM
# Notebook 02: Dataset Acquisition
# ============================================================

current_dir = Path.cwd()

if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

# Core directories
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

SEN1_DIR = RAW_DIR / "sen1floods11"
SPLITS_DIR = SEN1_DIR / "splits"

# New RoadFlood-VLM directories
HANDLABELED_DIR = SEN1_DIR / "hand_labeled"
PROTOTYPE_DIR = SEN1_DIR / "prototype"

S1_DIR = HANDLABELED_DIR / "S1Hand"
S2_DIR = HANDLABELED_DIR / "S2Hand"
LABEL_DIR = HANDLABELED_DIR / "LabelHand"
JRC_DIR = HANDLABELED_DIR / "JRCWaterHand"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
MAP_DIR = OUTPUT_DIR / "maps"

# Create directories if needed
required_directories = [
    HANDLABELED_DIR,
    PROTOTYPE_DIR,
    S1_DIR,
    S2_DIR,
    LABEL_DIR,
    JRC_DIR,
    TABLE_DIR,
    FIGURE_DIR,
    MAP_DIR,
]

for directory in required_directories:
    directory.mkdir(parents=True, exist_ok=True)

print("=" * 72)
print("ROADFLOOD-VLM DATASET ACQUISITION INITIALIZATION")
print("=" * 72)

print(f"\nProject root      : {PROJECT_ROOT}")
print(f"Sen1Floods11 root : {SEN1_DIR}")
print(f"Official splits   : {SPLITS_DIR}")
print(f"Prototype data    : {PROTOTYPE_DIR}")

print("\nHAND-LABELED COMPONENT DIRECTORIES")
print("-" * 72)
print(f"S1 imagery        : {S1_DIR}")
print(f"S2 imagery        : {S2_DIR}")
print(f"Flood labels      : {LABEL_DIR}")
print(f"Permanent water   : {JRC_DIR}")

print("\nENVIRONMENT")
print("-" * 72)
print(f"Python version    : {sys.version.split()[0]}")

print("\n" + "=" * 72)
print("NOTEBOOK 02 INITIALIZATION COMPLETE")
print("=" * 72)

ROADFLOOD-VLM DATASET ACQUISITION INITIALIZATION

Project root      : /home/adjeiowusu1/myproject/ResilientVLM
Sen1Floods11 root : /home/adjeiowusu1/myproject/ResilientVLM/data/raw/sen1floods11
Official splits   : /home/adjeiowusu1/myproject/ResilientVLM/data/raw/sen1floods11/splits
Prototype data    : /home/adjeiowusu1/myproject/ResilientVLM/data/raw/sen1floods11/prototype

HAND-LABELED COMPONENT DIRECTORIES
------------------------------------------------------------------------
S1 imagery        : /home/adjeiowusu1/myproject/ResilientVLM/data/raw/sen1floods11/hand_labeled/S1Hand
S2 imagery        : /home/adjeiowusu1/myproject/ResilientVLM/data/raw/sen1floods11/hand_labeled/S2Hand
Flood labels      : /home/adjeiowusu1/myproject/ResilientVLM/data/raw/sen1floods11/hand_labeled/LabelHand
Permanent water   : /home/adjeiowusu1/myproject/ResilientVLM/data/raw/sen1floods11/hand_labeled/JRCWaterHand

ENVIRONMENT
------------------------------------------------------------------------
Python 

In [2]:
# ============================================================
# CELL 2: LOAD OFFICIAL SEN1FLOODS11 SPLITS
# ============================================================

print("=" * 72)
print("OFFICIAL SPLIT TABLE LOADING")
print("=" * 72)

corrected_split_path = (
    TABLE_DIR
    / "sen1floods11_corrected_official_splits.csv"
)

print(f"\nSplit table path : {corrected_split_path}")
print(f"File exists      : {corrected_split_path.exists()}")

if not corrected_split_path.exists():
    raise FileNotFoundError(
        "The corrected official split table was not found. "
        "Run Notebook 01 through the corrected split cell first."
    )

official_splits_df = pd.read_csv(corrected_split_path)

required_columns = {
    "s1_filename",
    "label_filename",
    "split",
    "scene_id",
    "country_prefix",
}

missing_columns = (
    required_columns
    - set(official_splits_df.columns)
)

if missing_columns:
    raise ValueError(
        "The corrected split table is missing required columns: "
        f"{sorted(missing_columns)}"
    )

print("\nSPLIT TABLE SUMMARY")
print("-" * 72)

print(f"Total records     : {len(official_splits_df):,}")
print(
    f"Unique scenes     : "
    f"{official_splits_df['scene_id'].nunique():,}"
)

print("\nRecords by split:")

print(
    official_splits_df["split"]
    .value_counts()
    .to_string()
)

print("\nRecords by country:")

print(
    official_splits_df["country_prefix"]
    .value_counts()
    .to_string()
)

print("\nFirst five records:")

print(
    official_splits_df.head()
    .to_string(index=False)
)

print("\n" + "=" * 72)
print("OFFICIAL SPLIT TABLE LOADED SUCCESSFULLY")
print("=" * 72)

OFFICIAL SPLIT TABLE LOADING

Split table path : /home/adjeiowusu1/myproject/ResilientVLM/outputs/tables/sen1floods11_corrected_official_splits.csv
File exists      : True

SPLIT TABLE SUMMARY
------------------------------------------------------------------------
Total records     : 446
Unique scenes     : 446

Records by split:
split
train         252
test           90
validation     89
bolivia        15

Records by country:
country_prefix
USA          69
India        68
Paraguay     67
Ghana        53
Sri-Lanka    42
Mekong       30
Spain        30
Pakistan     28
Somalia      26
Nigeria      18
Bolivia      15

First five records:
            s1_filename             label_filename split     scene_id country_prefix
Ghana_103272_S1Hand.tif Ghana_103272_LabelHand.tif train Ghana_103272          Ghana
 Ghana_24858_S1Hand.tif  Ghana_24858_LabelHand.tif train  Ghana_24858          Ghana
Ghana_147015_S1Hand.tif Ghana_147015_LabelHand.tif train Ghana_147015          Ghana
Ghana_953791_S1H

In [3]:
# ============================================================
# CELL 3: SELECT PROTOTYPE TRAINING SCENES
# ============================================================

print("=" * 72)
print("PROTOTYPE SCENE SELECTION")
print("=" * 72)

TARGET_COUNTRIES = [
    "USA",
    "India",
    "Paraguay",
    "Ghana",
]

SELECTION_SEED = 42

training_df = official_splits_df[
    official_splits_df["split"] == "train"
].copy()

prototype_records = []

for country in TARGET_COUNTRIES:

    country_train_df = training_df[
        training_df["country_prefix"] == country
    ].copy()

    if country_train_df.empty:
        raise ValueError(
            f"No training scenes were found for {country}."
        )

    selected_row = (
        country_train_df
        .sample(
            n=1,
            random_state=SELECTION_SEED,
        )
        .iloc[0]
    )

    prototype_records.append(
        {
            "country": country,
            "scene_id": selected_row["scene_id"],
            "split": selected_row["split"],
            "s1_filename": selected_row["s1_filename"],
            "label_filename": selected_row["label_filename"],
        }
    )

prototype_selection_df = pd.DataFrame(
    prototype_records
)

print("\nSELECTED PROTOTYPE SCENES")
print("-" * 72)

print(
    prototype_selection_df.to_string(
        index=False
    )
)

prototype_selection_path = (
    TABLE_DIR
    / "roadflood_vlm_prototype_scene_selection.csv"
)

prototype_selection_df.to_csv(
    prototype_selection_path,
    index=False,
)

print("\nSelection saved to:")
print(prototype_selection_path)

print("\n" + "=" * 72)
print("PROTOTYPE SCENE SELECTION COMPLETE")
print("=" * 72)

PROTOTYPE SCENE SELECTION

SELECTED PROTOTYPE SCENES
------------------------------------------------------------------------
 country        scene_id split                s1_filename                label_filename
     USA       USA_58086 train       USA_58086_S1Hand.tif       USA_58086_LabelHand.tif
   India    India_698338 train    India_698338_S1Hand.tif    India_698338_LabelHand.tif
Paraguay Paraguay_224845 train Paraguay_224845_S1Hand.tif Paraguay_224845_LabelHand.tif
   Ghana     Ghana_26376 train     Ghana_26376_S1Hand.tif     Ghana_26376_LabelHand.tif

Selection saved to:
/home/adjeiowusu1/myproject/ResilientVLM/outputs/tables/roadflood_vlm_prototype_scene_selection.csv

PROTOTYPE SCENE SELECTION COMPLETE


In [4]:
# ============================================================
# CELL 4: BUILD AND VERIFY PROTOTYPE ACQUISITION MANIFEST
# ============================================================

import requests
import pandas as pd
from pathlib import Path

print("=" * 72)
print("PROTOTYPE ACQUISITION MANIFEST")
print("=" * 72)

BUCKET_NAME = "sen1floods11"
PUBLIC_BASE_URL = f"https://storage.googleapis.com/{BUCKET_NAME}"

COMPONENT_CONFIG = {
    "S1Hand": {
        "folder": "S1Hand",
        "suffix": "S1Hand",
        "local_root": S1_DIR,
    },
    "S2Hand": {
        "folder": "S2Hand",
        "suffix": "S2Hand",
        "local_root": S2_DIR,
    },
    "LabelHand": {
        "folder": "LabelHand",
        "suffix": "LabelHand",
        "local_root": LABEL_DIR,
    },
    "JRCWaterHand": {
        "folder": "JRCWaterHand",
        "suffix": "JRCWaterHand",
        "local_root": JRC_DIR,
    },
}

manifest_records = []

for _, scene_row in prototype_selection_df.iterrows():

    scene_id = scene_row["scene_id"]
    country = scene_row["country"]

    for component, config in COMPONENT_CONFIG.items():

        filename = f"{scene_id}_{config['suffix']}.tif"

        object_name = (
            "v1.1/data/flood_events/HandLabeled/"
            f"{config['folder']}/{filename}"
        )

        public_url = f"{PUBLIC_BASE_URL}/{object_name}"

        local_scene_dir = (
            PROTOTYPE_DIR
            / scene_id
        )

        local_scene_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        local_path = (
            local_scene_dir
            / filename
        )

        manifest_records.append(
            {
                "country": country,
                "scene_id": scene_id,
                "split": scene_row["split"],
                "component": component,
                "filename": filename,
                "object_name": object_name,
                "public_url": public_url,
                "local_path": str(local_path),
            }
        )

prototype_manifest_df = pd.DataFrame(
    manifest_records
)

print("\n1. MANIFEST SUMMARY")
print("-" * 72)

print(f"Scenes selected    : {prototype_manifest_df['scene_id'].nunique():,}")
print(f"Components/scene   : {len(COMPONENT_CONFIG):,}")
print(f"Total objects      : {len(prototype_manifest_df):,}")

print("\nObjects by component:")

print(
    prototype_manifest_df["component"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# Verify remote objects with HEAD requests
# ------------------------------------------------------------

print("\n2. REMOTE OBJECT VERIFICATION")
print("-" * 72)

verification_records = []

for index, row in prototype_manifest_df.iterrows():

    try:

        response = requests.head(
            row["public_url"],
            timeout=60,
            allow_redirects=True,
        )

        exists_remote = response.status_code == 200

        size_bytes = (
            int(response.headers.get("Content-Length", 0))
            if exists_remote
            else 0
        )

        content_type = response.headers.get(
            "Content-Type"
        )

        status = (
            "AVAILABLE"
            if exists_remote
            else f"HTTP {response.status_code}"
        )

    except requests.RequestException as exc:

        exists_remote = False
        size_bytes = 0
        content_type = None
        status = f"ERROR: {exc}"

    verification_records.append(
        {
            "remote_exists": exists_remote,
            "remote_size_bytes": size_bytes,
            "remote_size_mb": size_bytes / (1024 ** 2),
            "content_type": content_type,
            "verification_status": status,
        }
    )

    print(
        f"{row['scene_id']:<18} "
        f"{row['component']:<14} "
        f"[{status:<12}] "
        f"{size_bytes / (1024 ** 2):>8.2f} MB"
    )

verification_df = pd.DataFrame(
    verification_records
)

prototype_manifest_df = pd.concat(
    [
        prototype_manifest_df.reset_index(drop=True),
        verification_df.reset_index(drop=True),
    ],
    axis=1,
)

# ------------------------------------------------------------
# Validation summary
# ------------------------------------------------------------

print("\n3. VERIFICATION SUMMARY")
print("-" * 72)

available_count = int(
    prototype_manifest_df["remote_exists"].sum()
)

missing_count = (
    len(prototype_manifest_df)
    - available_count
)

total_download_mb = (
    prototype_manifest_df["remote_size_mb"].sum()
)

print(f"Objects available : {available_count:,}")
print(f"Objects missing   : {missing_count:,}")
print(f"Expected download : {total_download_mb:,.2f} MB")

if missing_count > 0:

    print("\nMissing or inaccessible objects:")

    missing_df = prototype_manifest_df[
        ~prototype_manifest_df["remote_exists"]
    ]

    print(
        missing_df[
            [
                "scene_id",
                "component",
                "object_name",
                "verification_status",
            ]
        ].to_string(index=False)
    )

# ------------------------------------------------------------
# Save verified manifest
# ------------------------------------------------------------

verified_manifest_path = (
    TABLE_DIR
    / "roadflood_vlm_verified_prototype_manifest.csv"
)

prototype_manifest_df.to_csv(
    verified_manifest_path,
    index=False,
)

print("\n4. MANIFEST OUTPUT")
print("-" * 72)

print(f"Saved to : {verified_manifest_path}")

print("\n" + "=" * 72)

if missing_count == 0:
    print("ALL PROTOTYPE OBJECTS VERIFIED")
else:
    print("PROTOTYPE OBJECT VERIFICATION REQUIRES ATTENTION")

print("=" * 72)

PROTOTYPE ACQUISITION MANIFEST

1. MANIFEST SUMMARY
------------------------------------------------------------------------
Scenes selected    : 4
Components/scene   : 4
Total objects      : 16

Objects by component:
component
S1Hand          4
S2Hand          4
LabelHand       4
JRCWaterHand    4

2. REMOTE OBJECT VERIFICATION
------------------------------------------------------------------------
USA_58086          S1Hand         [AVAILABLE   ]     1.41 MB


USA_58086          S2Hand         [AVAILABLE   ]     2.02 MB
USA_58086          LabelHand      [AVAILABLE   ]     0.00 MB


USA_58086          JRCWaterHand   [AVAILABLE   ]     0.00 MB
India_698338       S1Hand         [AVAILABLE   ]     1.58 MB


India_698338       S2Hand         [AVAILABLE   ]     2.34 MB
India_698338       LabelHand      [AVAILABLE   ]     0.01 MB


India_698338       JRCWaterHand   [AVAILABLE   ]     0.00 MB
Paraguay_224845    S1Hand         [AVAILABLE   ]     0.68 MB


Paraguay_224845    S2Hand         [AVAILABLE   ]     0.81 MB
Paraguay_224845    LabelHand      [AVAILABLE   ]     0.00 MB


Paraguay_224845    JRCWaterHand   [AVAILABLE   ]     0.00 MB
Ghana_26376        S1Hand         [AVAILABLE   ]     1.71 MB


Ghana_26376        S2Hand         [AVAILABLE   ]     2.30 MB
Ghana_26376        LabelHand      [AVAILABLE   ]     0.00 MB


Ghana_26376        JRCWaterHand   [AVAILABLE   ]     0.00 MB

3. VERIFICATION SUMMARY
------------------------------------------------------------------------
Objects available : 16
Objects missing   : 0
Expected download : 12.86 MB

4. MANIFEST OUTPUT
------------------------------------------------------------------------
Saved to : /home/adjeiowusu1/myproject/ResilientVLM/outputs/tables/roadflood_vlm_verified_prototype_manifest.csv

ALL PROTOTYPE OBJECTS VERIFIED


In [5]:
# ============================================================
# CELL 5: DOWNLOAD AND VALIDATE PROTOTYPE RASTERS
# ============================================================

import hashlib
import requests
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

print("=" * 72)
print("PROTOTYPE RASTER DOWNLOAD")
print("=" * 72)


def file_md5(file_path, chunk_size=1024 * 1024):
    """
    Calculate the MD5 checksum of a local file.
    """
    md5_hash = hashlib.md5()

    with open(file_path, "rb") as file_handle:
        while chunk := file_handle.read(chunk_size):
            md5_hash.update(chunk)

    return md5_hash.hexdigest()


def download_file_with_progress(
    url,
    output_path,
    expected_size_bytes=None,
    chunk_size=1024 * 1024,
):
    """
    Download a file with a progress bar.

    Existing files are skipped when their size matches the
    expected remote size.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists():

        existing_size = output_path.stat().st_size

        if (
            expected_size_bytes is not None
            and existing_size == expected_size_bytes
        ):
            return {
                "status": "SKIPPED",
                "local_size_bytes": existing_size,
            }

        output_path.unlink()

    temporary_path = output_path.with_suffix(
        output_path.suffix + ".part"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    with requests.get(
        url,
        stream=True,
        timeout=180,
    ) as response:

        response.raise_for_status()

        response_size = int(
            response.headers.get(
                "Content-Length",
                expected_size_bytes or 0,
            )
        )

        with open(temporary_path, "wb") as file_handle:

            with tqdm(
                total=response_size,
                unit="B",
                unit_scale=True,
                desc=output_path.name,
                leave=False,
            ) as progress_bar:

                for chunk in response.iter_content(
                    chunk_size=chunk_size
                ):

                    if not chunk:
                        continue

                    file_handle.write(chunk)
                    progress_bar.update(len(chunk))

    temporary_path.replace(output_path)

    return {
        "status": "DOWNLOADED",
        "local_size_bytes": output_path.stat().st_size,
    }


# ------------------------------------------------------------
# 1. Download all verified prototype objects
# ------------------------------------------------------------

print("\n1. DOWNLOADING FILES")
print("-" * 72)

download_results = []

for _, row in prototype_manifest_df.iterrows():

    local_path = Path(row["local_path"])
    expected_size = int(row["remote_size_bytes"])

    try:

        result = download_file_with_progress(
            url=row["public_url"],
            output_path=local_path,
            expected_size_bytes=expected_size,
        )

        local_size = result["local_size_bytes"]

        size_match = (
            local_size == expected_size
            if expected_size > 0
            else local_size > 0
        )

        checksum = (
            file_md5(local_path)
            if local_path.exists()
            else None
        )

        final_status = (
            "VALID"
            if local_path.exists() and size_match
            else "INVALID"
        )

        print(
            f"{row['scene_id']:<18} "
            f"{row['component']:<14} "
            f"[{result['status']:<10}] "
            f"{local_size / (1024 ** 2):>8.3f} MB "
            f"[{final_status}]"
        )

        download_results.append(
            {
                "download_status": result["status"],
                "local_exists": local_path.exists(),
                "local_size_bytes": local_size,
                "size_match": size_match,
                "md5": checksum,
                "validation_status": final_status,
                "download_error": None,
            }
        )

    except Exception as exc:

        print(
            f"{row['scene_id']:<18} "
            f"{row['component']:<14} "
            f"[FAILED] {exc}"
        )

        download_results.append(
            {
                "download_status": "FAILED",
                "local_exists": local_path.exists(),
                "local_size_bytes": (
                    local_path.stat().st_size
                    if local_path.exists()
                    else 0
                ),
                "size_match": False,
                "md5": None,
                "validation_status": "INVALID",
                "download_error": str(exc),
            }
        )


# ------------------------------------------------------------
# 2. Add results to manifest
# ------------------------------------------------------------

download_results_df = pd.DataFrame(download_results)

prototype_download_df = pd.concat(
    [
        prototype_manifest_df.reset_index(drop=True),
        download_results_df.reset_index(drop=True),
    ],
    axis=1,
)


# ------------------------------------------------------------
# 3. Validate by scene and component
# ------------------------------------------------------------

print("\n2. DOWNLOAD VALIDATION SUMMARY")
print("-" * 72)

valid_count = int(
    (
        prototype_download_df["validation_status"]
        == "VALID"
    ).sum()
)

invalid_count = (
    len(prototype_download_df)
    - valid_count
)

print(f"Files expected    : {len(prototype_download_df):,}")
print(f"Files valid       : {valid_count:,}")
print(f"Files invalid     : {invalid_count:,}")
print(
    f"Local total size  : "
    f"{prototype_download_df['local_size_bytes'].sum() / (1024 ** 2):,.2f} MB"
)

scene_validation = (
    prototype_download_df
    .pivot_table(
        index=["country", "scene_id"],
        columns="component",
        values="validation_status",
        aggfunc="first",
    )
)

print("\nValidation by scene:")

print(scene_validation.to_string())


# ------------------------------------------------------------
# 4. Detect partial download files
# ------------------------------------------------------------

partial_files = list(
    PROTOTYPE_DIR.rglob("*.part")
)

print("\n3. PARTIAL FILE CHECK")
print("-" * 72)

print(f"Partial files found : {len(partial_files):,}")

for partial_file in partial_files:
    print(f"  - {partial_file}")


# ------------------------------------------------------------
# 5. Save final acquisition manifest
# ------------------------------------------------------------

final_manifest_path = (
    TABLE_DIR
    / "roadflood_vlm_prototype_download_manifest.csv"
)

prototype_download_df.to_csv(
    final_manifest_path,
    index=False,
)

print("\n4. FINAL MANIFEST")
print("-" * 72)

print(f"Saved to : {final_manifest_path}")


# ------------------------------------------------------------
# 6. Final status
# ------------------------------------------------------------

print("\n" + "=" * 72)

if invalid_count == 0 and len(partial_files) == 0:
    print("PROTOTYPE DATASET DOWNLOAD COMPLETE")
else:
    print("PROTOTYPE DATASET DOWNLOAD REQUIRES ATTENTION")

print("=" * 72)

PROTOTYPE RASTER DOWNLOAD

1. DOWNLOADING FILES
------------------------------------------------------------------------


/home/adjeiowusu1/myproject/ResilientVLM/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


USA_58086_S1Hand.tif:   0%|                                                                                              | 0.00/1.47M [00:00<?, ?B/s]

USA_58086_S1Hand.tif:  71%|████████████████████████████████████████████████████████████▍                        | 1.05M/1.47M [00:00<00:00, 7.44MB/s]

USA_58086          S1Hand         [DOWNLOADED]    1.406 MB [VALID]


...

Ghana_26376        S2Hand         [DOWNLOADED]    2.304 MB [VALID]


Ghana_26376_LabelHand.tif:   0%|                                                                                         | 0.00/1.01k [00:00<?, ?B/s]

Ghana_26376        LabelHand      [DOWNLOADED]    0.001 MB [VALID]


Ghana_26376_JRCWaterHand.tif:   0%|                                                                                        | 0.00/835 [00:00<?, ?B/s]

Ghana_26376        JRCWaterHand   [DOWNLOADED]    0.001 MB [VALID]

2. DOWNLOAD VALIDATION SUMMARY
------------------------------------------------------------------------
Files expected    : 16
Files valid       : 16
Files invalid     : 0
Local total size  : 12.86 MB

Validation by scene:
component                JRCWaterHand LabelHand S1Hand S2Hand
country  scene_id                                            
Ghana    Ghana_26376            VALID     VALID  VALID  VALID
India    India_698338           VALID     VALID  VALID  VALID
Paraguay Paraguay_224845        VALID     VALID  VALID  VALID
USA      USA_58086              VALID     VALID  VALID  VALID

3. PARTIAL FILE CHECK
------------------------------------------------------------------------
Partial files found : 0

4. FINAL MANIFEST
------------------------------------------------------------------------
Saved to : /home/adjeiowusu1/myproject/ResilientVLM/outputs/tables/roadflood_vlm_prototype_download_manifest.csv

PROTOTYPE D

In [6]:
# ============================================================
# CELL 6: VALIDATE RASTER METADATA AND GEOSPATIAL ALIGNMENT
# ============================================================

import rasterio
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 72)
print("PROTOTYPE RASTER GEOSPATIAL VALIDATION")
print("=" * 72)

# ------------------------------------------------------------
# 1. Read metadata for every downloaded raster
# ------------------------------------------------------------

metadata_records = []

for _, row in prototype_download_df.iterrows():

    raster_path = Path(row["local_path"])

    with rasterio.open(raster_path) as src:

        metadata_records.append(
            {
                "country": row["country"],
                "scene_id": row["scene_id"],
                "component": row["component"],
                "filename": raster_path.name,
                "crs": str(src.crs),
                "width": src.width,
                "height": src.height,
                "band_count": src.count,
                "dtype": ",".join(src.dtypes),
                "nodata": src.nodata,
                "resolution_x": src.res[0],
                "resolution_y": src.res[1],
                "left": src.bounds.left,
                "bottom": src.bounds.bottom,
                "right": src.bounds.right,
                "top": src.bounds.top,
                "transform": str(src.transform),
            }
        )

raster_metadata_df = pd.DataFrame(metadata_records)


# ------------------------------------------------------------
# 2. Display metadata by scene
# ------------------------------------------------------------

print("\n1. RASTER METADATA BY SCENE")
print("-" * 72)

for scene_id in prototype_selection_df["scene_id"]:

    scene_metadata = raster_metadata_df[
        raster_metadata_df["scene_id"] == scene_id
    ]

    print(f"\nSCENE: {scene_id}")

    print(
        scene_metadata[
            [
                "component",
                "crs",
                "width",
                "height",
                "band_count",
                "dtype",
                "nodata",
                "resolution_x",
                "resolution_y",
            ]
        ].to_string(index=False)
    )


# ------------------------------------------------------------
# 3. Compare each component against S1Hand
# ------------------------------------------------------------

print("\n2. GEOSPATIAL ALIGNMENT COMPARISON")
print("-" * 72)

alignment_records = []

for scene_id in prototype_selection_df["scene_id"]:

    scene_metadata = raster_metadata_df[
        raster_metadata_df["scene_id"] == scene_id
    ].copy()

    reference = scene_metadata[
        scene_metadata["component"] == "S1Hand"
    ].iloc[0]

    for _, row in scene_metadata.iterrows():

        crs_match = row["crs"] == reference["crs"]

        shape_match = (
            row["width"] == reference["width"]
            and row["height"] == reference["height"]
        )

        resolution_match = (
            np.isclose(
                row["resolution_x"],
                reference["resolution_x"],
            )
            and
            np.isclose(
                row["resolution_y"],
                reference["resolution_y"],
            )
        )

        bounds_match = np.allclose(
            [
                row["left"],
                row["bottom"],
                row["right"],
                row["top"],
            ],
            [
                reference["left"],
                reference["bottom"],
                reference["right"],
                reference["top"],
            ],
        )

        transform_match = (
            row["transform"] == reference["transform"]
        )

        fully_aligned = all(
            [
                crs_match,
                shape_match,
                resolution_match,
                bounds_match,
                transform_match,
            ]
        )

        alignment_records.append(
            {
                "scene_id": scene_id,
                "component": row["component"],
                "crs_match": crs_match,
                "shape_match": shape_match,
                "resolution_match": resolution_match,
                "bounds_match": bounds_match,
                "transform_match": transform_match,
                "fully_aligned": fully_aligned,
            }
        )

alignment_df = pd.DataFrame(alignment_records)

print(
    alignment_df.to_string(index=False)
)


# ------------------------------------------------------------
# 4. Inspect band and label value distributions
# ------------------------------------------------------------

print("\n3. RASTER VALUE SUMMARY")
print("-" * 72)

value_records = []

for _, row in prototype_download_df.iterrows():

    raster_path = Path(row["local_path"])

    with rasterio.open(raster_path) as src:

        raster_data = src.read()

        valid_values = raster_data[
            np.isfinite(raster_data)
        ]

        if src.nodata is not None:
            valid_values = valid_values[
                valid_values != src.nodata
            ]

        unique_values = np.unique(valid_values)

        record = {
            "scene_id": row["scene_id"],
            "component": row["component"],
            "minimum": (
                float(valid_values.min())
                if valid_values.size
                else np.nan
            ),
            "maximum": (
                float(valid_values.max())
                if valid_values.size
                else np.nan
            ),
            "mean": (
                float(valid_values.mean())
                if valid_values.size
                else np.nan
            ),
            "unique_count": len(unique_values),
            "unique_values_preview": (
                unique_values[:20].tolist()
            ),
        }

        value_records.append(record)

value_summary_df = pd.DataFrame(value_records)

print(
    value_summary_df[
        [
            "scene_id",
            "component",
            "minimum",
            "maximum",
            "mean",
            "unique_count",
            "unique_values_preview",
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 5. Detailed LabelHand class distributions
# ------------------------------------------------------------

print("\n4. LABELHAND CLASS DISTRIBUTIONS")
print("-" * 72)

label_distribution_records = []

label_rows = prototype_download_df[
    prototype_download_df["component"] == "LabelHand"
]

for _, row in label_rows.iterrows():

    label_path = Path(row["local_path"])

    with rasterio.open(label_path) as src:
        label_array = src.read(1)

    values, counts = np.unique(
        label_array,
        return_counts=True,
    )

    total_pixels = counts.sum()

    print(f"\nSCENE: {row['scene_id']}")

    for value, count in zip(values, counts):

        percentage = (
            100 * count / total_pixels
        )

        print(
            f"  Value {value:<6} "
            f"Pixels: {count:>8,} "
            f"Percentage: {percentage:>8.4f}%"
        )

        label_distribution_records.append(
            {
                "scene_id": row["scene_id"],
                "label_value": value,
                "pixel_count": count,
                "percentage": percentage,
            }
        )

label_distribution_df = pd.DataFrame(
    label_distribution_records
)


# ------------------------------------------------------------
# 6. Save outputs
# ------------------------------------------------------------

metadata_output = (
    TABLE_DIR
    / "roadflood_vlm_prototype_raster_metadata.csv"
)

alignment_output = (
    TABLE_DIR
    / "roadflood_vlm_prototype_alignment.csv"
)

value_output = (
    TABLE_DIR
    / "roadflood_vlm_prototype_value_summary.csv"
)

label_output = (
    TABLE_DIR
    / "roadflood_vlm_prototype_label_distribution.csv"
)

raster_metadata_df.to_csv(
    metadata_output,
    index=False,
)

alignment_df.to_csv(
    alignment_output,
    index=False,
)

value_summary_df.to_csv(
    value_output,
    index=False,
)

label_distribution_df.to_csv(
    label_output,
    index=False,
)


# ------------------------------------------------------------
# 7. Final validation status
# ------------------------------------------------------------

print("\n5. FINAL VALIDATION STATUS")
print("-" * 72)

all_aligned = alignment_df[
    "fully_aligned"
].all()

print(
    f"All raster components fully aligned : "
    f"{all_aligned}"
)

print("\nOutput tables:")

print(f"  - {metadata_output}")
print(f"  - {alignment_output}")
print(f"  - {value_output}")
print(f"  - {label_output}")

print("\n" + "=" * 72)

if all_aligned:
    print("PROTOTYPE GEOSPATIAL VALIDATION PASSED")
else:
    print("PROTOTYPE GEOSPATIAL VALIDATION REQUIRES REVIEW")

print("=" * 72)

PROTOTYPE RASTER GEOSPATIAL VALIDATION

1. RASTER METADATA BY SCENE
------------------------------------------------------------------------

SCENE: USA_58086
   component       crs  width  height  band_count                                                                         dtype  nodata  resolution_x  resolution_y
      S1Hand EPSG:4326    512     512           2                                                               float32,float32     NaN       0.00009       0.00009
      S2Hand EPSG:4326    512     512          13 int16,int16,int16,int16,int16,int16,int16,int16,int16,int16,int16,int16,int16     0.0       0.00009       0.00009
   LabelHand EPSG:4326    512     512           1                                                                         int16     NaN       0.00009       0.00009
JRCWaterHand EPSG:4326    512     512           1                                                                         uint8     NaN       0.00009       0.00009

SCENE: India_698338


       scene_id    component    minimum     maximum        mean  unique_count                                                                                                                                                                                                                                                                                                                                                                                                        unique_values_preview
      USA_58086       S1Hand -54.735344   13.172750  -15.676058        402605          [-54.73534393310547, -53.67810821533203, -50.81943893432617, -50.02626419067383, -48.71110153198242, -48.58583450317383, -48.538551330566406, -48.409812927246094, -48.18490982055664, -47.99644088745117, -47.87603759765625, -47.8007926940918, -47.76300048828125, -47.743812561035156, -47.741641998291016, -47.5640754699707, -47.500057220458984, -47.44483947753906, -47.42902755737305, -47.32489776611328]
      USA_58086   

In [7]:
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\takyi\Desktop\Machine Learning Engineering\ResilientVLM"
)

search_terms = [
    "manifest",
    "selected",
    "download",
    "scene",
]

for path in PROJECT_ROOT.rglob("*"):
    if not path.is_file():
        continue

    name_lower = path.name.lower()

    if any(term in name_lower for term in search_terms):
        if path.suffix.lower() in {
            ".csv",
            ".json",
            ".jsonl",
            ".txt",
            ".parquet",
        }:
            print(path)